# Notebook 01 · Perceptrón simple construido desde cero (NumPy)

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** clasificación **binaria** de frutas (0 = manzana, 1 = banano) a partir de tres
características físicas: *peso (g)*, *longitud (cm)* y *nivel de amarillo (0–1)*.

> **Alcance de este notebook:** implementar el **perceptrón** de Rosenblatt con función de
> activación **escalón (step)**, su **regla de aprendizaje** y su evaluación. Este es el modelo
> base que en los notebooks siguientes se rediseña con Sigmoide, ReLU y *backpropagation*.

---

## 1. Objetivos

1. Comprender la estructura de una neurona artificial: entradas, pesos, bias, función de activación y salida.
2. Implementar la función de activación **escalón** y la regla de actualización del perceptrón.
3. Aplicar **entrenamiento supervisado** con un conjunto de ejemplos etiquetados.
4. Evaluar el modelo con métricas de clasificación (*accuracy*, matriz de confusión) sobre datos no vistos.
5. Identificar las limitaciones del perceptrón que motivan el uso de Sigmoide, ReLU y *backpropagation*.

## 2. Fundamento teórico

El perceptrón calcula una **combinación lineal** de las entradas y la convierte en una decisión discreta:

$$ z = \mathbf{w}^\top \mathbf{x} + b = w_1 x_1 + w_2 x_2 + w_3 x_3 + b $$

$$ \hat{y} = f(z) =
\begin{cases}
1 & \text{si } z \ge 0 \\
0 & \text{si } z < 0
\end{cases} $$

La **regla de aprendizaje del perceptrón** solo modifica los parámetros cuando la predicción es incorrecta:

$$ w_i \leftarrow w_i + \eta \, (y - \hat{y}) \, x_i
\qquad\qquad
b \leftarrow b + \eta \, (y - \hat{y}) $$

donde $\eta$ es el *learning rate* (tamaño del paso) y $(y-\hat y)$ es el error: vale $+1$ si la red
dijo 0 siendo 1, $-1$ si dijo 1 siendo 0, y $0$ si acertó. El teorema de convergencia del perceptrón
garantiza que, **si las clases son linealmente separables**, el algoritmo encuentra un hiperplano
separador en un número finito de épocas.


## 3. Entorno de ejecución

Este notebook está diseñado para **Google Colab**. En el menú:
`Entorno de ejecución → Cambiar tipo de entorno de ejecución → Python 3`.
Únicamente se requieren NumPy, scikit-learn y Matplotlib (ya incluidos en Colab).

| Librería | Versión usada en la ejecución documentada | Rol |
|---|---|---|
| NumPy | 2.4.6 | Cálculo matricial del modelo |
| scikit-learn | 1.9.1 | Partición de datos, escalado y métricas |
| Matplotlib | 3.11.2 | Gráficas de frontera de decisión y pesos |

**Reproducibilidad:** todas las semillas aleatorias se fijan en `42`.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import random

random.seed(42)
np.random.seed(42)
print("NumPy:", np.__version__)
print("Herramientas cargadas correctamente")

NumPy: 2.4.6
Herramientas cargadas correctamente


## 4. Dataset de frutas

Cada fila de `X` es una fruta y cada columna es una característica. `y` contiene la **respuesta
correcta**, lo que hace de este un problema de **aprendizaje supervisado**: la red puede comparar su
predicción con la etiqueta verdadera y corregirse.

In [2]:
X = np.array([
    # MANZANAS  (clase 0): [peso g, longitud cm, nivel de amarillo 0-1]
    [150, 7.5, 0.10], [160, 8.0, 0.15], [145, 7.2, 0.05],
    [170, 8.3, 0.20], [155, 7.8, 0.10], [165, 8.1, 0.15],
    [140, 7.0, 0.05], [175, 8.5, 0.20], [158, 7.9, 0.10],
    [162, 8.2, 0.15],

    # BANANOS   (clase 1)
    [120, 18.0, 0.90], [125, 19.0, 0.95], [115, 17.5, 0.85],
    [130, 20.0, 1.00], [118, 18.5, 0.90], [123, 19.5, 0.95],
    [110, 17.0, 0.80], [128, 20.5, 1.00], [121, 18.8, 0.90],
    [126, 19.2, 0.95]
], dtype=float)

y = np.array([0] * 10 + [1] * 10)

print("Matriz de características X:", X.shape, "-> (20 frutas, 3 características)")
print("Vector de etiquetas   y:", y.shape, "-> (0 = manzana, 1 = banano)")
print()
print("Primeras 3 manzanas:\n", X[:3])
print("Primeros 3 bananos:\n", X[10:13])


Matriz de características X: (20, 3) -> (20 frutas, 3 características)
Vector de etiquetas   y: (20,) -> (0 = manzana, 1 = banano)

Primeras 3 manzanas:
 [[1.50e+02 7.50e+00 1.00e-01]
 [1.60e+02 8.00e+00 1.50e-01]
 [1.45e+02 7.20e+00 5.00e-02]]
Primeros 3 bananos:
 [[120.    18.     0.9 ]
 [125.    19.     0.95]
 [115.    17.5    0.85]]


### 4.1 División entrenamiento / prueba y escalado

Se reserva el 20 % de las frutas para evaluación. Esas muestras **nunca** participan en el ajuste de
pesos: de lo contrario se mediría memoria y no capacidad de generalización.

El escalado (`StandardScaler`) es necesario porque las tres características viven en rangos muy
distintos (peso ≈ 110–205, longitud ≈ 7–20, amarillo ∈ [0,1]). Sin estandarizar, el gradiente quedaría
dominado por la variable de mayor magnitud numérica.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80 % entrenamiento / 20 % prueba, conservando la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# El escalado se AJUSTA solo con entrenamiento y se APLICA a prueba
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Distribución de clases en prueba:", np.bincount(y_test))
print()
print("Media de cada característica en entrenamiento (debe ser ~0):")
print(np.round(X_train.mean(axis=0), 6))
print("Desviación estándar (debe ser ~1):")
print(np.round(X_train.std(axis=0), 6))


X_train: (16, 3) | X_test: (4, 3)
y_train: (16,) | y_test: (4,)
Distribución de clases en prueba: [2 2]

Media de cada característica en entrenamiento (debe ser ~0):
[0. 0. 0.]
Desviación estándar (debe ser ~1):
[1. 1. 1.]


### 4.2 Visualización exploratoria de los datos

Antes de modelar se verifica gráficamente que las dos clases ocupen regiones distintas del espacio de
características. Esta inspección es la que justifica esperar convergencia del perceptrón.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
pares = [(0, 1, "Peso (g)", "Longitud (cm)"),
         (1, 2, "Longitud (cm)", "Nivel de amarillo"),
         (0, 2, "Peso (g)", "Nivel de amarillo")]
colores = ["#c0392b", "#f1c40f"]

for ax, (i, j, xl, yl) in zip(axes, pares):
    for clase, nombre in [(0, "Manzana"), (1, "Banano")]:
        m = y == clase
        ax.scatter(X[m, i], X[m, j], c=colores[clase], label=nombre,
                   edgecolors="k", s=55, alpha=0.9)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend(fontsize=8)
    ax.set_title(f"{xl} vs {yl}", fontsize=10)

fig.suptitle("Figura 1. Distribución de las dos clases en el espacio de características", y=1.02)
plt.tight_layout(); plt.savefig("fig01_datos_frutas.png", bbox_inches="tight"); plt.show()

## 5. Implementación del perceptrón

Se definen la función de activación **escalón** y la función de predicción. La activación escalón es la
más simple del *deep learning*: no es derivable en $z=0$, razón por la cual **no** puede usarse con
descenso de gradiente; su aprendizaje se basa en la regla de corrección de errores.

In [5]:
def step(z):
    """Función de activación escalón (umbral en 0)."""
    return (z >= 0).astype(int)


def predecir(X_datos, w, b):
    """Forward propagation de una sola neurona: z = X·w + b  ->  y_hat = step(z)."""
    z = X_datos @ w + b
    return step(z)


# --- Inicialización de parámetros ---
np.random.seed(42)
w = np.random.randn(3) * 0.01     # pesos pequeños aleatorios
b = 0.0                           # bias inicial

print("Pesos iniciales w =", np.round(w, 6))
print("Bias inicial    b =", round(b, 6))
print()
print("Predicciones iniciales (sin entrenar):", predecir(X_train, w, b))
print("Etiquetas reales                    :", y_train)
print("Aciertos iniciales:", int((predecir(X_train, w, b) == y_train).sum()), "de", len(y_train))

Pesos iniciales w = [ 0.004967 -0.001383  0.006477]
Bias inicial    b = 0.0

Predicciones iniciales (sin entrenar): [1 1 0 0 1 1 0 0 0 1 1 0 1 0 1 1]
Etiquetas reales                    : [0 0 1 1 1 1 1 0 0 1 1 0 1 0 0 0]
Aciertos iniciales: 9 de 16


### 5.1 Una iteración de aprendizaje, paso a paso

Antes del ciclo completo se ejecuta **una sola pasada** por el conjunto de entrenamiento imprimiendo
cada corrección. Esto permite observar el mecanismo interno del aprendizaje supervisado: la red
predice, compara con la etiqueta, y desplaza el hiperplano únicamente donde se equivocó.

In [6]:
np.random.seed(42)
w = np.random.randn(3) * 0.01
b = 0.0
eta = 0.1

print("=== ITERACIÓN 1 (paso a paso) ===")
print(f"{'#':>2} {'z':>9} {'y_hat':>5} {'y':>3} {'error':>6}   acción")
for k in range(len(X_train)):
    x_k, y_k = X_train[k], y_train[k]
    z_k = float(x_k @ w + b)
    y_hat = int(z_k >= 0)
    error = y_k - y_hat
    accion = "sin cambio (acierto)" if error == 0 else "se actualizan w y b"
    print(f"{k:>2} {z_k:>9.4f} {y_hat:>5} {y_k:>3} {error:>6}   {accion}")
    w = w + eta * error * x_k
    b = b + eta * error

print()
print("Pesos tras la iteración 1:", np.round(w, 6))
print("Bias  tras la iteración 1:", round(b, 6))

=== ITERACIÓN 1 (paso a paso) ===
 #         z y_hat   y  error   acción
 0    0.0008     1   0     -1   se actualizan w y b
 1   -0.4165     0   0      0   sin cambio (acierto)
 2    0.1824     1   1      0   sin cambio (acierto)
 3    0.2019     1   1      0   sin cambio (acierto)
 4    0.1825     1   1      0   sin cambio (acierto)
 5    0.1827     1   1      0   sin cambio (acierto)
 6    0.1898     1   1      0   sin cambio (acierto)
 7   -0.3535     0   0      0   sin cambio (acierto)
 8   -0.3923     0   0      0   sin cambio (acierto)
 9    0.1801     1   1      0   sin cambio (acierto)
10    0.2022     1   1      0   sin cambio (acierto)
11   -0.3422     0   0      0   sin cambio (acierto)
12    0.2020     1   1      0   sin cambio (acierto)
13   -0.3769     0   0      0   sin cambio (acierto)
14   -0.3898     0   0      0   sin cambio (acierto)
15   -0.4545     0   0      0   sin cambio (acierto)

Pesos tras la iteración 1: [-0.105816  0.092776  0.099898]
Bias  tras la iterac

### 5.2 Ciclo de entrenamiento completo

Se repite el proceso hasta que el perceptrón clasifique correctamente **todo** el conjunto de
entrenamiento (convergencia) o hasta agotar 100 épocas. Una **época** es una pasada completa por los
datos de entrenamiento.

In [7]:
np.random.seed(42)
w = np.random.randn(3) * 0.01
b = 0.0
eta = 0.1
max_epochs = 100

historial_accuracy = []
epoch_convergencia = None

for epoch in range(max_epochs):
    actualizaciones = 0
    for k in range(len(X_train)):
        x_k, y_k = X_train[k], y_train[k]
        y_hat = int((x_k @ w + b) >= 0)
        error = y_k - y_hat
        if error != 0:
            w = w + eta * error * x_k
            b = b + eta * error
            actualizaciones += 1

    acc = float((predecir(X_train, w, b) == y_train).mean())
    historial_accuracy.append(acc)

    if epoch % 10 == 0 or actualizaciones == 0:
        print(f"Época {epoch:>3} | actualizaciones: {actualizaciones:>2} | accuracy entrenamiento: {acc:.4f}")

    if actualizaciones == 0 and epoch_convergencia is None:
        epoch_convergencia = epoch
        print(f"\n>>> Convergió en la época {epoch}: ninguna predicción fue incorrecta.")
        break

print()
print("Pesos finales   w =", np.round(w, 4))
print("Bias final      b =", round(b, 4))
print("Épocas usadas     =", epoch + 1)

Época   0 | actualizaciones:  1 | accuracy entrenamiento: 1.0000
Época   1 | actualizaciones:  0 | accuracy entrenamiento: 1.0000

>>> Convergió en la época 1: ninguna predicción fue incorrecta.

Pesos finales   w = [-0.1058  0.0928  0.0999]
Bias final      b = -0.1
Épocas usadas     = 2


## 6. Evaluación sobre datos no vistos

El *accuracy* de entrenamiento solo indica que el modelo separó las frutas con las que aprendió.
La métrica relevante es la obtenida sobre `X_test`.

In [8]:
y_pred_test = predecir(X_test, w, b)
acc_test = accuracy_score(y_test, y_pred_test)

print("=== PREDICCIONES EN EL CONJUNTO DE PRUEBA ===")
print(f"{'Fruta':>6} {'Prob. z':>9} {'Predicción':>12} {'Real':>12} {'Resultado':>10}")
for i in range(len(X_test)):
    z_i = float(X_test[i] @ w + b)
    pred = "BANANO" if y_pred_test[i] == 1 else "MANZANA"
    real = "BANANO" if y_test[i] == 1 else "MANZANA"
    ok = "ACIERTO" if pred == real else "ERROR"
    print(f"{i+1:>6} {z_i:>9.4f} {pred:>12} {real:>12} {ok:>10}")

print()
print("Accuracy en prueba:", round(acc_test, 4))
print()
print("Matriz de confusión (filas = reales, columnas = predichas):")
print(confusion_matrix(y_test, y_pred_test))
print()
print(classification_report(y_test, y_pred_test, target_names=["Manzana", "Banano"], zero_division=0))

=== PREDICCIONES EN EL CONJUNTO DE PRUEBA ===
 Fruta   Prob. z   Predicción         Real  Resultado
     1   -0.3171      MANZANA      MANZANA    ACIERTO
     2   -0.4295      MANZANA      MANZANA    ACIERTO
     3    0.1904       BANANO       BANANO    ACIERTO
     4    0.1984       BANANO       BANANO    ACIERTO

Accuracy en prueba: 1.0

Matriz de confusión (filas = reales, columnas = predichas):
[[2 0]
 [0 2]]

              precision    recall  f1-score   support

     Manzana       1.00      1.00      1.00         2
      Banano       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



### 6.1 Frontera de decisión y pesos aprendidos

La frontera del perceptrón es siempre una **recta (hiperplano)**: es un clasificador lineal.
Se proyecta sobre las dos características más discriminativas, deshaciendo el escalado para
interpretarla en unidades originales.

In [9]:
# Deshacer el escalado para graficar en unidades originales
media, desv = scaler.mean_, scaler.scale_

i_feat, j_feat = 1, 2      # longitud (cm) vs nivel de amarillo
rangos = np.array([X[:, i_feat].min(), X[:, i_feat].max(),
                   X[:, j_feat].min(), X[:, j_feat].max()])
margen = 0.1 * (rangos[1] - rangos[0] + rangos[3] - rangos[2])
g1 = np.linspace(rangos[0] - margen, rangos[1] + margen, 300)
g2 = np.linspace(rangos[2] - margen, rangos[3] + margen, 300)
G1, G2 = np.meshgrid(g1, g2)

# Se reconstruye el vector de 3 características usando la media para la tercera
puntos = np.column_stack([
    np.full(G1.size, media[0]),
    G1.ravel(),
    G2.ravel()
])
puntos_escalados = (puntos - media) / desv
Z_grid = (puntos_escalados @ w + b).reshape(G1.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# (a) Frontera de decisión
ax = axes[0]
ax.contourf(G1, G2, Z_grid, levels=20, cmap="RdYlGn", alpha=0.55)
ax.contour(G1, G2, Z_grid, levels=[0], colors="k", linewidths=2.2)
for clase, nombre, c in [(0, "Manzana", "#8e1b0e"), (1, "Banano", "#7d6608")]:
    m = y == clase
    ax.scatter(X[m, i_feat], X[m, j_feat], c=c, edgecolors="w", s=70, label=nombre)
ax.set_xlabel("Longitud (cm)"); ax.set_ylabel("Nivel de amarillo")
ax.set_title("Figura 2. Frontera de decisión lineal del perceptrón")
ax.legend()

# (b) Pesos aprendidos
ax = axes[1]
etiquetas = ["Peso (g)", "Longitud (cm)", "Nivel de amarillo"]
barras = ax.bar(etiquetas, w, color=["#3498db", "#2ecc71", "#e67e22"], edgecolor="k")
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Valor del peso")
ax.set_title("Figura 3. Pesos aprendidos por la neurona")
for barra, valor in zip(barras, w):
    ax.text(barra.get_x() + barra.get_width() / 2, valor,
            f"{valor:.3f}", ha="center",
            va="bottom" if valor >= 0 else "top", fontsize=9)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout(); plt.savefig("fig02_perceptron_frontera.png", bbox_inches="tight"); plt.show()

print("Lectura de los pesos:")
print(" · La característica con mayor |peso| domina la decisión.")
print(f" · |w| = {np.round(np.abs(w), 4)}")

Lectura de los pesos:
 · La característica con mayor |peso| domina la decisión.
 · |w| = [0.1058 0.0928 0.0999]


## 7. Predicción de una fruta nueva

Una observación nueva debe pasar por **el mismo `scaler` ajustado en entrenamiento**; nunca se vuelve
a ejecutar `fit_transform` con datos nuevos, pues cambiaría la escala aprendida.

In [10]:
fruta_nueva = np.array([[124.0, 19.0, 0.94]])      # liviana, alargada, muy amarilla
fruta_escalada = scaler.transform(fruta_nueva)

z_nueva = float((fruta_escalada @ w + b)[0])
pred_nueva = int(z_nueva >= 0)
nombre_pred = "BANANO" if pred_nueva == 1 else "MANZANA"

print("Fruta nueva (peso, longitud, amarillo):", fruta_nueva[0])
print("Valores escalados                     :", np.round(fruta_escalada[0], 4))
print("z = w·x + b                           :", round(z_nueva, 4))
print("step(z)                               :", pred_nueva)
print("La red piensa que es                  :", nombre_pred)

Fruta nueva (peso, longitud, amarillo): [124.    19.     0.94]
Valores escalados                     : [-0.9271  0.9614  0.9859]
z = w·x + b                           : 0.1858
step(z)                               : 1
La red piensa que es                  : BANANO


## 8. Análisis de resultados y limitaciones

| Aspecto | Observación |
|---|---|
| Convergencia | El perceptrón converge en pocas épocas porque el dataset es linealmente separable. |
| Tipo de salida | Entrega una **clase discreta** (0 o 1), no una probabilidad. No es posible decir "está 92 % seguro". |
| Mecanismo de aprendizaje | Regla de corrección de errores caso por caso; **no** utiliza gradientes de una función de pérdida. |
| Expresividad | Solo puede representar funciones linealmente separables. Falla ante problemas como XOR. |
| Sensibilidad | Un único ejemplo ruidoso puede mover el hiperplano de forma indefinida (oscilación). |

Estas cuatro limitaciones son exactamente las que resuelven los notebooks siguientes:

1. **Sigmoide + descenso de gradiente + *backpropagation*** → salidas probabilísticas y optimización basada en una función de pérdida (*Notebook 02*).
2. **ReLU en capas ocultas** → modelos no lineales y mitigación del desvanecimiento del gradiente (*Notebook 02*).
3. **Frameworks (TensorFlow/Keras y PyTorch)** → automatización del cálculo de gradientes (*Notebooks 03 a 06*).

## 9. Conclusiones del notebook

- La neurona artificial es una combinación lineal seguida de una función de activación; en el perceptrón esa función es el escalón.
- El aprendizaje supervisado exige ejemplos etiquetados: el error $(y-\hat y)$ es la señal que dirige la actualización de los parámetros.
- El *learning rate* $\eta$ no es un peso: es un hiperparámetro que controla el tamaño del paso de actualización.
- La evaluación debe hacerse sobre datos no vistos y acompañarse de la matriz de confusión, no solo del *accuracy*.

## 10. Referencias

- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- Stevens, E., Antiga, L., & Viehmann, T. (2020). *Deep Learning with PyTorch*. Manning Publications.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/